# Explore post-processed Amazon Reviews 2023 data

Sanity-checks on the output of `02_preprocess_amazon2023.ipynb`: split sizes, id consistency, interaction distributions, and a peek at the simulator jsonl.

In [ ]:
import os
import json

import pandas as pd
import config 

In [ ]:
# --- config: point this at the same OUTPUT_DIR / DATA_DIR used in notebook 02 ---
DATA_DIR = os.path.expanduser("~/blob/raw_datasets/amazon-2023/All_Beauty")
OUTPUT_DIR = os.path.join(DATA_DIR, "chatbot")

## Load outputs

In [ ]:
df_train = pd.read_csv(os.path.join(OUTPUT_DIR, "train.tsv"))
df_valid = pd.read_csv(os.path.join(OUTPUT_DIR, "valid.tsv"))
df_test = pd.read_csv(os.path.join(OUTPUT_DIR, "test.tsv"))
user_history = pd.read_csv(os.path.join(OUTPUT_DIR, "user_history.tsv"))
products = pd.read_feather(os.path.join(OUTPUT_DIR, "products.ftr"))

with open(os.path.join(DATA_DIR, "map.json")) as f:
    id_maps = json.load(f)

simulator_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("simulator_test_data_")]
simulator_path = os.path.join(OUTPUT_DIR, simulator_files[0])
simulator_data = [json.loads(line) for line in open(simulator_path)]

print("train:", df_train.shape)
print("valid:", df_valid.shape)
print("test:", df_test.shape)
print("user_history (train+valid):", user_history.shape)
print("products:", products.shape)
print("users in map:", len(id_maps["user"]))
print("items in map:", len(id_maps["item"]))
print("simulator records:", len(simulator_data))

## Sanity checks

In [ ]:
# leave-one-out: every user should appear exactly once in valid and once in test
valid_counts = df_valid["user_id"].value_counts()
test_counts = df_test["user_id"].value_counts()
print("users with != 1 valid row:", (valid_counts != 1).sum())
print("users with != 1 test row:", (test_counts != 1).sum())

In [ ]:
# item ids in the splits should all exist in the product table
known_ids = set(products["id"])
for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    missing = (~df["item_id"].isin(known_ids)).sum()
    print(f"{name}: {missing} item_ids missing from products table")

In [ ]:
# id range should match the map sizes
print("max user_id in train:", df_train["user_id"].max(), "vs users in map:", len(id_maps["user"]))
print("max item_id in train:", df_train["item_id"].max(), "vs items in map:", len(id_maps["item"]))

## Interaction distributions

In [ ]:
all_inter = pd.concat([df_train, df_valid, df_test])
print("interactions per user:\n", all_inter.groupby("user_id").size().describe())
print("\ninteractions per item:\n", all_inter.groupby("item_id").size().describe())

## Product table

In [ ]:
print("visited_num stats:")
print(products["visited_num"].describe())
print("\nmost-visited products:")
print(products.sort_values("visited_num", ascending=False)[["title", "visited_num"]].head(10))

In [ ]:
print("category distribution:")
print(products["category"].value_counts().head(20))
print(f"\nmissing/placeholder description: {(products['description'] == 'No description').mean():.1%}")
print(f"missing price: {products['price'].isna().mean():.1%}")

## Simulator jsonl peek

In [ ]:
for rec in simulator_data[:5]:
    print("history:", rec["history"][:200])
    print("target: ", rec["target"])
    print()